In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

import numpy as np
import pandas as pd
import os
import random
import time
import math
import json
from sklearn.model_selection import train_test_split
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
from transformers import BertConfig, BertModel
from transformers import RobertaConfig, RobertaModel

In [2]:

class TextDataset(Dataset):
    def __init__(self, data, flag='train'):
        self.data = data
        self.flag = flag

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        ind = torch.tensor(row['id'], dtype=torch.int)
        text = torch.tensor(row['text'][:2048], dtype=torch.long)
        if self.flag != 'test':
            label = torch.tensor(row['label'], dtype=torch.long)
            return text, label, ind
        else:
            return text, ind

class DataFactory(object):
    def __init__(self, path, max_len=512, flag='train'):
        self.path = path
        self.flag = flag
        self.max_len = max_len
        self.df = pd.read_json(self.path, lines=True)

    def collate_fn(self, batch):
        texts, labels, ids = zip(*batch)
        padded_texts = pad_sequence(texts, batch_first=True, padding_value=0)
        mask = (padded_texts != 0).long() 
        labels = torch.stack(labels)
        ids = torch.stack(ids)
        return padded_texts, mask, labels, ids

    def collate_fn_test(self, batch):
        texts, ids = zip(*batch)
        padded_texts = pad_sequence(texts, batch_first=True, padding_value=0)
        mask = (padded_texts != 0).long() 
        ids = torch.stack(ids)
        return padded_texts, mask, ids

    def get_dataloader(self, batch_size=32):
        if self.flag != 'test':
            train_data, val_data = train_test_split(self.df, test_size=0.3, random_state=42)
            train_dataset = TextDataset(train_data, flag='train')
            val_dataset = TextDataset(val_data, flag='val')
            train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=self.collate_fn)
            val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=self.collate_fn)
            return train_loader, val_loader
        elif self.flag == 'semi':
            semi_dataset = TextDataset(self.df, flag='semi')
            semi_loader = DataLoader(semi_dataset, batch_size=batch_size, shuffle=True, collate_fn=self.collate_fn)
            return semi_loader
        else:
            test_dataset = TextDataset(self.df, flag='test')
            test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=self.collate_fn_test)
            return test_loader
            
        

In [3]:
'''Maximum Mean Discrepancy Loss'''
def mmd_loss(a,b):
    if a.size(0)<2 or b.size(0)<2:
        return torch.tensor(0., device=a.device)
    mu_a, mu_b = a.mean(0), b.mean(0)
    return torch.sum((mu_a-mu_b)**2)

In [4]:
class Classifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, hidden_dim, num_layers, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.positional_encoding = nn.Parameter(torch.zeros(1, 2048, embed_dim))
        self.transformer = nn.Transformer(
            d_model=embed_dim, nhead=num_heads, num_encoder_layers=num_layers, dim_feedforward=hidden_dim
        )
        self.fc = nn.Linear(embed_dim, num_classes+2)
        self.proj = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(inplace=True),
            nn.Linear(embed_dim, hidden_dim)
        )

    def forward(self, x, x_mask):
        # x: (batch_size, seq_len)
        B, L = x.shape
        x = self.embedding(x) + self.positional_encoding[:, :L, :]
        x = x.permute(1, 0, 2)  # Transformer expects (seq_len, batch_size, embed_dim)
        x = self.transformer(x, x)  # Using the same input as both src and tgt
        x = x.mean(dim=0)  # Pooling over the sequence dimension
        logits = self.fc(x)
        z = self.proj(x)              # (B, proj_dim)
        z = F.normalize(z, dim=1)
        return logits, z


class BERTClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, hidden_dim, num_layers, num_classes=2):
        super().__init__()
        # self.embedding = nn.Embedding(vocab_size, embed_dim)
        # self.positional_encoding = nn.Parameter(torch.zeros(1, 2048, embed_dim))
        self.fc = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(128, embed_dim)
        )
        
        self.res = nn.Linear(embed_dim, 128)
        
        self.classifier = nn.Linear(embed_dim, num_classes)
        
        self.proj = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(inplace=True),
            nn.Linear(embed_dim, embed_dim)
        )
        config = BertConfig(
            vocab_size=vocab_size,
            hidden_size=embed_dim,
            num_hidden_layers=num_layers,
            num_attention_heads=num_heads,
            intermediate_size=hidden_dim,
            max_position_embeddings=4096,
            output_hidden_states=True
        )
        self.bert = BertModel(config)


    def forward(self, x, x_mask):
        if x.size(1) == 0:
            # Return dummy logits and embeddings with appropriate batch size
            batch_size = x.size(0)
            dummy_logits = torch.zeros(batch_size, self.fc[-1].out_features, device=x.device)
            dummy_z = torch.zeros(batch_size, self.proj[-1].out_features, device=x.device)
            return dummy_logits, dummy_z

        # x_embed = self.embedding(x) + self.positional_encoding[:, :x.size(1), :]
        out = self.bert(input_ids=x,
                        attention_mask=x_mask)
        cls_vec = out.hidden_states[-1][:, 0, :]
        logits = self.classifier(self.fc(cls_vec) + cls_vec)
        # z = self.proj(cls_vec)              
        # z = F.normalize(z, dim=1)
        return logits, cls_vec

    def extract_cls_from_tokens(self, dataloader, device):
        self.eval()
        features, labels, ids = [], [], []
        with torch.no_grad():
            for x, x_mask, y, ids_batch in dataloader:
                # x_mask = (x != 0).long()
                x, x_mask = x.to(device), x_mask.to(device)
                _, cls_vec = self(x, x_mask)
                features.append(cls_vec.cpu())
                labels.append(y.cpu())
                ids.extend([i.item() if isinstance(i, torch.Tensor) else int(i) for i in ids_batch])
        return torch.cat(features), torch.cat(labels), ids


class RoBERTaClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, hidden_dim, num_layers, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.positional_encoding = nn.Parameter(torch.zeros(1, 2048, embed_dim))
        self.fc = nn.Linear(embed_dim, num_classes)
        self.proj = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(inplace=True),
            nn.LayerNorm(embed_dim),
            nn.Dropout(0.1),
            nn.Linear(embed_dim, hidden_dim)
        )
        config = RobertaConfig(
            vocab_size=vocab_size,
            hidden_size=embed_dim,
            num_hidden_layers=num_layers,
            num_attention_heads=num_heads,
            intermediate_size=hidden_dim,
            max_position_embeddings=4096,
        )
        self.roberta = RobertaModel(config)

    def forward(self, x, x_mask):
        if x.size(1) == 0:
            # Return dummy logits and embeddings with appropriate batch size
            batch_size = x.size(0)
            dummy_logits = torch.zeros(batch_size, self.fc.out_features, device=x.device)
            dummy_z = torch.zeros(batch_size, self.proj[-1].out_features, device=x.device)
            return dummy_logits, dummy_z

        # x_embed = self.embedding(x) + self.positional_encoding[:, :x.size(1), :]
        out = self.roberta(input_ids=x,
                        attention_mask=x_mask)
        sequence_output = out.last_hidden_state
        cls_vec = sequence_output[:, 0, :]
        logits = self.fc(cls_vec)
        z = self.proj(cls_vec)              
        z = F.normalize(z, dim=1)
        return logits, z

class LSTM_BERT_Classifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, lstm_hidden_dim, num_heads, hidden_dim, num_layers, num_classes=2):
        super().__init__()

        # Step 1: Embedding + LSTM
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, lstm_hidden_dim, batch_first=True, bidirectional=True)

        # Projection to match BERT/Transformer input size
        self.project = nn.Linear(2 * lstm_hidden_dim, embed_dim)

        # Step 2: BERT-style Transformer encoder
        config = BertConfig(
            vocab_size=vocab_size,
            hidden_size=embed_dim,
            num_hidden_layers=num_layers,
            num_attention_heads=num_heads,
            intermediate_size=hidden_dim,
            max_position_embeddings=4096,
            output_hidden_states=True
        )
        self.bert = BertModel(config)

        # Step 3: Classifier head
        self.classifier = nn.Linear(embed_dim, num_classes)

        # Optional projection head for MMD / SupCon
        self.proj = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(inplace=True),
            nn.Linear(embed_dim, embed_dim)
        )

    def forward(self, x, x_mask):
        B, L = x.shape

        # Step 1: Embedding + LSTM
        x_embed = self.embedding(x)                        # (B, L, E)
        x_lstm, _ = self.lstm(x_embed)                    # (B, L, 2H)
        z_lstm = x_lstm.mean(dim=1)                       # (B, 2H) for MMD (optional)

        # Step 2: Project and feed to BERT
        x_proj = self.project(x_lstm)                     # (B, L, E)
        out = self.bert(inputs_embeds=x_proj, attention_mask=x_mask)
        cls_vec = out.last_hidden_state[:, 0, :]          # (B, E)

        # Step 3: Classification
        logits = self.classifier(cls_vec)

        # Optional projection for SupCon or MMD
        z = self.proj(cls_vec)
        z = F.normalize(z, dim=1)

        return logits, z, z_lstm

# Initialize the model
class Main(object):
    def __init__(self, configs):
        random.seed(configs.seed)
        np.random.seed(configs.seed)
        torch.manual_seed(configs.seed)
        self.configs = configs
        self.name = configs.name
        self.embed_dim = configs.embed_dim
        self.num_heads = configs.num_heads
        self.hidden_dim = configs.hidden_dim
        self.lstm_hidden_dim = configs.lstm_hidden_dim
        self.num_layers = configs.num_layers
        self.num_classes = configs.num_classes
        self.criterion = nn.CrossEntropyLoss()
        self.data1 = DataFactory(configs.path1, flag='train')
        self.data2 = DataFactory(configs.path2, flag='val')
        self.data_test = DataFactory(configs.test_path, flag='test')
        self.trainloader_1, self.valloader_1 = self.data1.get_dataloader(batch_size=configs.batch_size1)
        self.trainloader_2, self.valloader_2 = self.data2.get_dataloader(batch_size=configs.batch_size2)
        self.testloader = self.data_test.get_dataloader()
        self.second_trainloader = None
        self.second_valloader = None
        
        self.vocab_size = 17212
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        # self.model = Classifier(self.vocab_size, self.embed_dim, self.num_heads, self.hidden_dim, self.num_layers, self.num_classes).to(self.device)
        self.model = BERTClassifier(self.vocab_size, self.embed_dim, self.num_heads, self.hidden_dim, self.num_layers, self.num_classes).to(self.device)
        # self.model = RoBERTaClassifier(self.vocab_size, self.embed_dim, self.num_heads, self.hidden_dim, self.num_layers, self.num_classes).to(self.device)
        # self.model = LSTM_BERT_Classifier(self.vocab_size, self.embed_dim, self.lstm_hidden_dim, self.num_heads, self.hidden_dim, self.num_layers, self.num_classes).to(self.device)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-4)
        self.num_epochs = configs.num_epochs
        self.metric1 = accuracy_score
        self.metric2 = f1_score
        self.tau = configs.tau

    def __save__(self):
        path = os.path.join(f'../checkpoints/{self.name}')
        if not os.path.exists(path):
            os.makedirs(path)
        torch.save(self.model.state_dict(), f'{path}/{self.name}.pt')

    def __load__(self):
        path = os.path.join(f'../checkpoints/{self.name}')
        if not os.path.exists(path):
            os.makedirs(path)
        self.model.load_state_dict(torch.load(f'{path}/{self.name}.pt', map_location=self.device))

    # def supcon_loss(self, z: torch.Tensor, y: torch.Tensor, T: float = 0.07):
    #     sim = torch.matmul(z, z.T) / T             # (N, N)
    #     sim_max, _ = sim.max(dim=1, keepdim=True)
    #     sim = sim - sim_max.detach()
    #     N         = len(z)
    #     eye_bool  = torch.eye(N, dtype=torch.bool, device=z.device)
    #     mask      = ~eye_bool                  # True 表示非自己
    
    #     exp_sim   = torch.exp(sim) * mask      # 元素乘，float*bool 会自动转 float
    #     pos_mask  = y.unsqueeze(0) == y.unsqueeze(1)  # (N, N) 布尔型
    
    #     pos_sim   = exp_sim * pos_mask         # 只保留同标签对
    #     loss_i    = -torch.log(pos_sim.sum(1) / exp_sim.sum(1))
    #     return loss_i.mean

    def supcon_loss(self, z: torch.Tensor, y: torch.Tensor, T: float = 0.07) -> torch.Tensor:
        """
        Computes the Supervised Contrastive Loss (SupCon) for a batch.
        
        Args:
            z: Tensor of shape (N, D), L2-normalized projection vectors.
            y: LongTensor of shape (N,), integer class labels.
            T: Float, temperature parameter.
        
        Returns:
            A scalar Tensor containing the mean SupCon loss over the batch.
        """
        N = z.size(0)
        
        # 1) Pairwise cosine similarities scaled by temperature → (N, N)
        sim = torch.matmul(z, z.T) / T
        
        # 2) Numerical stability: subtract max per row
        sim_max, _ = sim.max(dim=1, keepdim=True)
        sim = sim - sim_max.detach()
        
        # 3) Exponentiate and zero out self-similarities on the diagonal
        exp_sim = torch.exp(sim)
        eye_mask = torch.eye(N, dtype=torch.bool, device=z.device)
        exp_sim = exp_sim.masked_fill(eye_mask, 0.0)
        
        # 4) Build mask for positives: same label across batch
        pos_mask = y.unsqueeze(0) == y.unsqueeze(1)  # shape (N, N)
        
        # 5) Sum of positive similarities and sum of all similarities
        pos_sum = (exp_sim * pos_mask.float()).sum(dim=1)
        all_sum = exp_sim.sum(dim=1)
        
        # 6) Avoid log(0) by clamping to a small epsilon
        eps = 1e-6
        pos_sum = pos_sum.clamp_min(eps)
        all_sum = all_sum.clamp_min(eps)
        
        # 7) Compute per-sample loss and then average
        loss_i = -torch.log(pos_sum / all_sum)
        return loss_i.mean()



    def train(self):
        '''
        Train the model with CE loss and optional MMD domain adaptation.
        Combines domain1 and domain2 batches, pads inputs, computes:
        - Cross-entropy loss (always)
        - MMD loss (after epoch 6) for domain alignment
        Applies early stopping with patience. Saves and reloads best model.
        '''
        loader_1 = self.trainloader_1
        loader_2 = self.trainloader_2
        min_loss = math.inf
        patience = 3
        for epoch in range(self.num_epochs):
            self.model.train()
            epoch_loss = []
            epoch_acc = []
            epoch_f1 = []
            for (x1, x1_mask, y1, ind1), (x2, x2_mask, y2, ind2) in tqdm(zip(loader_1, loader_2)):
                x1, x1_mask, y1, ind1 = x1.to(self.device), x1_mask.to(self.device), y1.to(self.device), ind1.to(self.device)
                x2, x2_mask, y2, ind2 = x2.to(self.device), x2_mask.to(self.device), y2.to(self.device), ind2.to(self.device)
                domain = torch.cat([torch.zeros_like(y1), torch.ones_like(y2)], dim=0)

                L = max(x1.size(1), x2.size(1))
                # pad 第二维 (seq_len) 到 L
                x1 = F.pad(x1, (0, L - x1.size(1)))       # (left, right) for dim=1
                x2 = F.pad(x2, (0, L - x2.size(1)))
                m1 = F.pad(x1_mask, (0, L - x1_mask.size(1)))
                m2 = F.pad(x2_mask, (0, L - x2_mask.size(1)))
                
                x  = torch.cat([x1, x2], dim=0)
                mask = torch.cat([m1, m2], dim=0)
                y = torch.cat([y1, y2], dim=0)
                
                self.optimizer.zero_grad()
                outputs, feats = self.model(x, mask)
                pred = torch.argmax(outputs, dim=1)
                loss_ce = self.criterion(outputs, y)
                loss_supcon = self.supcon_loss(outputs, y)

                z_d1_ai = feats[(domain==0)&(y==1)]
                z_d2_ai = feats[(domain==1)&(y==1)]
                z_d1_h  = feats[(domain==0)&(y==0)]
                z_d2_h  = feats[(domain==1)&(y==0)]
                loss_mmd_ai    = mmd_loss(z_d1_ai, z_d2_ai)
                loss_mmd_human = mmd_loss(z_d1_h,  z_d2_h)
                loss_mmd = loss_mmd_ai + loss_mmd_human
                
                if epoch <= 6:
                    loss = loss_ce
                else:
                    loss = loss_ce + self.configs.lambda_mmd*loss_mmd
                # loss = loss_ce + self.tau*loss_supcon
                loss.backward()
                self.optimizer.step()
                epoch_loss.append(loss.item())
                epoch_acc.append(self.metric1(pred.detach().cpu(), y.detach().cpu()))
                epoch_f1.append(self.metric2(pred.detach().cpu(), y.detach().cpu(), average='macro'))

            epoch_loss = np.mean(epoch_loss)
            epoch_acc = np.mean(epoch_acc)
            epoch_f1 = np.mean(epoch_f1)
            print(f"Epoch {epoch + 1:>3}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}, F1: {epoch_f1:.4f}")

            vali_loss = self.validation()
            self.model.train()
            if vali_loss < min_loss:
                min_loss = vali_loss
                self.__save__()
            else:
                patience -= 1

            if not patience:
                break

        self.__load__()


    def validation(self):
        loader_1 = self.valloader_1
        loader_2 = self.valloader_2
        self.model.eval()
        with torch.no_grad():
            vali_loss = []
            vali_acc = []
            vali_f1 = []
            for (x1, x1_mask, y1, ind1), (x2, x2_mask, y2, ind2) in tqdm(zip(loader_1, loader_2)):
                x1, x1_mask, y1, ind1 = x1.to(self.device), x1_mask.to(self.device), y1.to(self.device), ind1.to(self.device)
                x2, x2_mask, y2, ind2 = x2.to(self.device), x2_mask.to(self.device), y2.to(self.device), ind2.to(self.device)
                domain = torch.cat([torch.zeros_like(y1), torch.ones_like(y2)], dim=0)

                L = max(x1.size(1), x2.size(1))
                # pad 第二维 (seq_len) 到 L
                x1 = F.pad(x1, (0, L - x1.size(1)))       # (left, right) for dim=1
                x2 = F.pad(x2, (0, L - x2.size(1)))
                m1 = F.pad(x1_mask, (0, L - x1_mask.size(1)))
                m2 = F.pad(x2_mask, (0, L - x2_mask.size(1)))
                
                x  = torch.cat([x1, x2], dim=0)
                mask = torch.cat([m1, m2], dim=0)
                y = torch.cat([y1, y2], dim=0)
                outputs, feats = self.model(x, mask)
                pred = torch.argmax(outputs, dim=1)
                loss = self.criterion(outputs, y)
                vali_loss.append(loss.item())
                vali_acc.append(self.metric1(pred.detach().cpu(), y.detach().cpu()))
                vali_f1.append(self.metric2(pred.detach().cpu(), y.detach().cpu(), average='macro'))
                
            vali_loss = np.mean(vali_loss)
            vali_acc = np.mean(vali_acc)
            vali_f1 = np.mean(vali_f1)
            print(f"Validation Loss: {vali_loss:.4f}, Accuracy: {vali_acc:.4f}, F1: {vali_f1:.4f}")
            return vali_loss




    def collect_pseudo_labels(self, threshold=0.95):
        '''
        Collects high-confidence pseudo-labeled samples from validation data.

        Runs the model in evaluation mode over both validation loaders (domain 1 and 2),
        and generates pseudo-labels for samples where the model's prediction confidence
        (softmax probability) exceeds the given threshold and matches the true label.

        Returns:
            pseudo_samples_1 (list): High-confidence samples from domain 1.
            pseudo_samples_2 (list): High-confidence samples from domain 2.
            low_conf_samples_1 (list): Low-confidence or misclassified samples from domain 1.
            low_conf_samples_2 (list): Low-confidence or misclassified samples from domain 2.
        '''
        import json
        loader_1 = self.valloader_1
        loader_2 = self.valloader_2
        self.model.eval()
        pseudo_samples_1, pseudo_samples_2 = [], []
        low_conf_samples_1, low_conf_samples_2 = [], []

        with torch.no_grad():
            for (x1, x1_mask, y1, ind1), (x2, x2_mask, y2, ind2) in tqdm(zip(loader_1, loader_2)):
                x1, x1_mask, ind1 = x1.to(self.device), x1_mask.to(self.device), ind1.to(self.device)
                x2, x2_mask, ind2 = x2.to(self.device), x2_mask.to(self.device), ind2.to(self.device)

                L = max(x1.size(1), x2.size(1))
                x1 = F.pad(x1, (0, L - x1.size(1)))
                x2 = F.pad(x2, (0, L - x2.size(1)))
                m1 = F.pad(x1_mask, (0, L - x1_mask.size(1)))
                m2 = F.pad(x2_mask, (0, L - x2_mask.size(1)))

                x = torch.cat([x1, x2], dim=0)
                mask = torch.cat([m1, m2], dim=0)
                ind = torch.cat([ind1, ind2], dim=0)

                outputs, _ = self.model(x, mask)
                prob = torch.softmax(outputs, dim=1)
                max_prob, pseudo_label = prob.max(dim=1)

                y = torch.cat([y1, y2], dim=0).to(self.device)  # true labels for all samples

                for i in range(len(x)):
                    pred = pseudo_label[i]
                    item = (x[i].cpu(), pred.cpu(), ind[i].cpu())
                    if i < len(x1):  # domain 1
                        if max_prob[i] > threshold and pred == y[i]:
                            pseudo_samples_1.append(item)
                        else:
                            low_conf_samples_1.append(item)
                    else:  # domain 2
                        if max_prob[i] > threshold and pred == y[i]:
                            pseudo_samples_2.append(item)
                        else:
                            low_conf_samples_2.append(item)

        print(f"Collected {len(pseudo_samples_1)} + {len(pseudo_samples_2)} pseudo-labeled samples, {len(low_conf_samples_1)} + {len(low_conf_samples_2)} remaining in val.")

        return pseudo_samples_1, pseudo_samples_2, low_conf_samples_1, low_conf_samples_2
    

    def test(self):
        '''take the test set and save the prediction using current model for test set, save the result in csv'''
        test_loader = self.testloader
        self.model.eval()
        self.__load__()
        all_ids, all_preds = [], []
        with torch.no_grad():
            for x, x_mask, ind in tqdm(test_loader):
                x, x_mask = x.to(self.device), x_mask.to(self.device)
                outputs, _ = self.model(x, x_mask)
                pred = torch.argmax(outputs, dim=1).cpu()
                all_ids.append(ind)
                all_preds.append(pred)
        all_ids = torch.cat(all_ids).numpy()
        all_preds = torch.cat(all_preds).numpy()
        df = pd.DataFrame({
            "id":   all_ids,
            "class": all_preds
        })
        df.to_csv(self.configs.save_path, index=False)
        print(f"Saved → {self.configs.save_path}")

    def train_on_cls_embeddings(self, epochs=15, batch_size=32, lr=1e-5, use_mixup=True):
        """
        Fine-tunes only the classifier head using precomputed [CLS] embeddings of new data set.
        """
        device = self.device
        min_loss = math.inf
        patience = 3
        self.model.to(device)
        self.model.train()

        # Freeze non-head parts
        for p in self.model.bert.parameters():
            p.requires_grad = False
        for p in self.model.proj.parameters():
            p.requires_grad = False

        train_loader = self.second_trainloader
        val_loader = self.second_valloader

        # Train only fc and classifier layers
        params = list(self.model.fc.parameters()) + list(self.model.classifier.parameters())
        optimizer = torch.optim.Adam(params, lr=lr)
        criterion = nn.CrossEntropyLoss()

        def one_hot(labels, num_classes):
            return torch.nn.functional.one_hot(labels, num_classes=num_classes).float()
        # Mixup oversampling
        def mixup_embeddings(x, y, alpha=0.2):
            lam = np.random.beta(alpha, alpha)
            index = torch.randperm(x.size(0)).to(x.device)
            mixed_x = lam * x + (1 - lam) * x[index]
            mixed_y = lam * y + (1 - lam) * y[index]
            return mixed_x, mixed_y

        for epoch in range(epochs):
            self.model.train()
            epoch_loss, epoch_acc, epoch_f1 = [], [], []

            for cls_vec, y, _ in tqdm(train_loader):
                cls_vec, y = cls_vec.to(device), y.to(device)

                optimizer.zero_grad()

                if use_mixup:
                    y_onehot = one_hot(y, self.configs.num_classes).to(device)
                    cls_vec, y_mix = mixup_embeddings(cls_vec, y_onehot)
                    cls_out = self.model.fc(cls_vec)
                    logits = self.model.classifier(cls_out + cls_vec)
                    loss = torch.sum(-y_mix * torch.nn.functional.log_softmax(logits, dim=1), dim=1).mean()
                    pred = torch.argmax(logits, dim=1)
                    true_y = torch.argmax(y_mix, dim=1)
                else:
                    cls_out = self.model.fc(cls_vec)
                    logits = self.model.classifier(cls_out + cls_vec)
                    loss = criterion(logits, y)
                    pred = torch.argmax(logits, dim=1)
                    true_y = y

                loss.backward()
                optimizer.step()

                epoch_loss.append(loss.item())
                epoch_acc.append(self.metric1(pred.cpu(), true_y.cpu()))
                epoch_f1.append(self.metric2(pred.cpu(), true_y.cpu(), average='macro'))

            print(f"[Head Epoch {epoch + 1}] Loss: {np.mean(epoch_loss):.4f}, "
                f"Acc: {np.mean(epoch_acc):.4f}, F1: {np.mean(epoch_f1):.4f}")

            val_loss = self.second_validation()
            self.model.train()

            if val_loss < min_loss:
                min_loss = val_loss
                self.__save__()
                patience = 3
            else:
                patience -= 1

            if patience == 0:
                print("Early stopping triggered.")
                break
        self.__load__()

        
    def second_validation(self):
        """
        Run full model on validation loader after taken out the psesudo data.
        Expects each batch to be (x, x_mask, y, ids).
        Returns val-loss for early stopping.
        """
        self.model.eval()
        device      = self.device
        val_loader  = self.second_valloader
        criterion   = nn.CrossEntropyLoss()

        total_loss, total_acc, total_f1 = [], [], []

        with torch.no_grad():
            for x, x_mask, y, _ in val_loader:
                x, x_mask, y = x.to(device), x_mask.to(device), y.to(device)

                # === Full forward pass (embedding + BERT + head) ===
                logits, _ = self.model(x, x_mask)

                loss = criterion(logits, y)
                pred = torch.argmax(logits, dim=1)

                total_loss.append(loss.item())
                total_acc.append(self.metric1(pred.cpu(), y.cpu()))
                total_f1.append(self.metric2(pred.cpu(), y.cpu(), average='macro'))

        avg_loss = np.mean(total_loss)
        avg_acc  = np.mean(total_acc)
        avg_f1   = np.mean(total_f1)

        print(f"[Second Val] Loss: {avg_loss:.4f}, Acc: {avg_acc:.4f}, F1: {avg_f1:.4f}")
        return avg_loss




In [5]:
class Config:
    lambda_mmd = 0.001
    name = 'BERT_mixup_semi'
    embed_dim = 256
    lstm_hidden_dim = 64
    num_heads = 16
    hidden_dim = 128
    num_layers = 2
    num_classes = 2
    path1 = '../data/raw/domain1_train_data.json'
    path2 = '../data/raw/domain2_train_data.json'
    test_path = '../data/raw/test_data.json'
    num_epochs = 15
    batch_size1 = 32
    batch_size2 = 32
    seed = 42
    tau=1
    save_path = "../results/mixup_pred.csv"

configs = Config()
main = Main(configs)

In [6]:
'''get raw data from data loader'''
def extract_raw_data(dataset):
    result = []
    for i in range(len(dataset)):
        text, label, idx = dataset[i]
        mask = (text != 0).long()
        result.append((text, label, idx))
    return result

In [7]:
def build_dataloader_from_pseudo(data_list, batch_size, collate_fn):
    class PseudoDataset(torch.utils.data.Dataset):
        def __init__(self, data):
            self.data = data  # (text, mask, label, id)

        def __len__(self):
            return len(self.data)

        def __getitem__(self, idx):
            text, label, idx_ = self.data[idx]
            return text, label, idx_ 

    dataset = PseudoDataset(data_list)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    return loader


In [8]:
class SecondPhaseDataset(Dataset):
    def __init__(self, features, labels, ids):
        self.features = torch.tensor(features, dtype=torch.float)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.ids = ids

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx], torch.tensor(self.ids[idx], dtype=torch.long)


In [9]:
main.train()

0it [00:00, ?it/s]

1it [00:00,  3.50it/s]

2it [00:00,  3.63it/s]

3it [00:00,  3.75it/s]

4it [00:01,  3.83it/s]

...

6it [00:00, 12.28it/s]

8it [00:00, 12.25it/s]

10it [00:00, 13.54it/s]

10it [00:00, 12.93it/s]

Validation Loss: 0.2974, Accuracy: 0.9141, F1: 0.8810


In [10]:

from torch.utils.data import Dataset, DataLoader
# collect pseudo data
pseudo_data_1, pseudo_data_2, val_data_1, val_data_2 = main.collect_pseudo_labels(threshold=0.99)
train_data_1 = extract_raw_data(main.trainloader_1.dataset)
train_data_2 = extract_raw_data(main.trainloader_2.dataset)

# create dataloader for new data set
combined_train_data_1 = train_data_1 + pseudo_data_1
combined_train_data_2 = train_data_2 + pseudo_data_2
temp_loader_1 = build_dataloader_from_pseudo(combined_train_data_1, batch_size=32, collate_fn=main.data2.collate_fn)
temp_loader_2 = build_dataloader_from_pseudo(combined_train_data_2, batch_size=32, collate_fn=main.data2.collate_fn)

# embeddings
feat_1, label_1, id_1 = main.model.extract_cls_from_tokens(temp_loader_1, main.device)
feat_2, label_2, id_2 = main.model.extract_cls_from_tokens(temp_loader_2, main.device)

features = torch.cat([feat_1, feat_2])
labels = torch.cat([label_1, label_2])
ids = id_1 + id_2

dataset = SecondPhaseDataset(features.numpy(), labels.numpy(), ids)
main.second_trainloader = DataLoader(dataset, batch_size=main.configs.batch_size1, shuffle=True)

val_data = val_data_1 + val_data_2
# Build validation loaders from remaining validation data
val_loader = build_dataloader_from_pseudo(val_data, batch_size=main.configs.batch_size1, collate_fn=main.data2.collate_fn)

# Assign to main class
main.second_valloader = val_loader

main.train_on_cls_embeddings()  # Phase 2 training with updated loaders



0it [00:00, ?it/s]

2it [00:00, 12.90it/s]

4it [00:00, 12.43it/s]

6it [00:00, 12.25it/s]

8it [00:00, 12.82it/s]

...

 70%|██████▉   | 101/145 [00:00<00:00, 239.25it/s]

 87%|████████▋ | 126/145 [00:00<00:00, 242.15it/s]

100%|██████████| 145/145 [00:00<00:00, 239.67it/s]

[Head Epoch 4] Loss: 0.0871, Acc: 0.9875, F1: 0.9689


[Second Val] Loss: 0.1869, Acc: 0.9955, F1: 0.9954
Early stopping triggered.


In [11]:
'''final validation on all data, including train and validation'''
train_data_1 = extract_raw_data(main.trainloader_1.dataset)
train_data_2 = extract_raw_data(main.trainloader_2.dataset)
val_data_1 = extract_raw_data(main.valloader_1.dataset)
val_data_2 = extract_raw_data(main.valloader_2.dataset)
full_data = train_data_1 + train_data_2 + val_data_1 + val_data_2
full_loader = build_dataloader_from_pseudo(full_data, configs.batch_size2, main.data2.collate_fn)
def evaluate_model(model, dataloader, criterion, device, source_name="all"):
    model.eval()
    all_preds, all_labels, all_ids = [], [], []
    misclassified = []

    total_loss = 0

    with torch.no_grad():
        for x, x_mask, y, ids in dataloader:
            x, x_mask, y = x.to(device), x_mask.to(device), y.to(device)
            outputs, _ = model(x, x_mask)
            loss = criterion(outputs, y)
            total_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_ids.extend(ids.cpu().numpy())

            for i in range(len(y)):
                true_label = y[i].item()
                pred_label = preds[i].item()
                sample_id = ids[i].item()
                if true_label != pred_label:
                    misclassified.append((sample_id, true_label, pred_label, source_name))

    from sklearn.metrics import accuracy_score, f1_score
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')

    print(f"\nEvaluation on {source_name} — Loss: {total_loss:.4f}, Accuracy: {acc:.4f}, F1: {f1:.4f}")
    print(f"Misclassified Samples: {len(misclassified)}")
    return

evaluate_model(main.model, full_loader, main.criterion, main.device)




Evaluation on all — Loss: 11.4726, Accuracy: 0.9782, F1: 0.9498
Misclassified Samples: 131


In [12]:
# main.test()